# 003 Handoffs

这是 LangChain Multi-agent 学习线的第三份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs

学习目标：

1. 理解 handoff 是 active agent / active step 的控制权转移
2. 区分 handoff 和 subagent-as-tool
3. 学会用 state 记录当前 step
4. 学会用 `Command(update=...)` 触发 handoff
5. 学会用 middleware 根据当前 step 动态切换 prompt 和 tools
6. 理解为什么 handoff 通常需要 checkpointer
7. 对比本仓库 Harness 的 coordinator / subagent delegate 模式

这一讲使用 fake model，但状态更新、handoff 和 graph 执行机制是真实的。

## 1. Handoff 和 Subagent 的区别

上一讲的 subagent 模式是：

```text
supervisor 调用 subagent tool
subagent 完成局部任务
结果返回 supervisor
supervisor 继续控制对话
```

handoff 模式是：

```text
当前 agent / step 收集到足够信息
触发状态更新
active step 切换到另一个 agent / step
新 step 接管后续对话行为
```

一句话：

```text
subagent 是委派；handoff 是交接。
```

In [27]:
from typing import Callable

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


def print_messages(result: dict) -> None:
    for message in result.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


## 2. 用 state 记录当前 step

handoff 的关键是状态。

下面定义一个 support workflow：

```text
triage
  -> 收集保修信息
  -> record_warranty_status(...)
  -> current_step = specialist
specialist
  -> 根据保修状态给解决方案
```

`current_step` 就是当前 active agent / active step。

In [28]:
class SupportState(AgentState):
    current_step: str
    warranty_status: str | None


## 3. 用 Command(update=...) 触发 handoff

工具可以返回 `Command`。

`Command(update=...)` 可以更新 graph state。

这里 `record_warranty_status` 会做两件事：

1. 写入 `warranty_status`
2. 把 `current_step` 改成 `specialist`

这就是 handoff 的核心。

In [29]:
@tool
def record_warranty_status(status: str, runtime: ToolRuntime) -> Command:
    """Record warranty status and transfer control to the specialist step."""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content="Warranty status recorded: " + status,
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "specialist",
        }
    )


@tool
def provide_solution(runtime: ToolRuntime) -> str:
    """Provide a solution after handoff to specialist."""
    return "specialist solution"


## 4. 根据当前 step 动态切换 prompt 和 tools

handoff 后，不只是状态变了。

模型看到的 prompt 和 tools 也应该变。

`triage` 阶段只暴露 `record_warranty_status`。

`specialist` 阶段只暴露 `provide_solution`。

In [30]:
@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    step = request.state.get("current_step", "triage")
    print("active step before model:", step)

    if step == "triage":
        request = request.override(
            system_prompt="你是售后 triage agent。先收集保修状态，然后调用 record_warranty_status。",
            tools=[record_warranty_status],
        )
    else:
        warranty_status = request.state.get("warranty_status")
        request = request.override(
            system_prompt="你是售后 specialist agent。当前保修状态：" + str(warranty_status),
            tools=[provide_solution],
        )

    return handler(request)


## 5. 跑通 handoff

fake model 第一轮会调用 `record_warranty_status`。

工具返回 `Command(update=...)` 后，第二轮模型调用前 active step 会变成 `specialist`。

In [31]:
handoff_model = ToolCallingFakeModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "record_warranty_status",
                    "args": {"status": "valid"},
                    "id": "call_1",
                }
            ],
        ),
        AIMessage(content="已转到 specialist，并根据有效保修状态给出解决方案。"),
    ]
)

handoff_agent = create_agent(
    model=handoff_model,
    tools=[record_warranty_status, provide_solution],
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)

handoff_result = handoff_agent.invoke(
    {"messages": [{"role": "user", "content": "我的保修仍然有效，需要售后支持。"}]},
    config={"configurable": {"thread_id": "handoff-demo"}},
)

print_messages(handoff_result)
print("current_step:", handoff_result.get("current_step"))
print("warranty_status:", handoff_result.get("warranty_status"))


active step before model: triage
active step before model: specialist
human 我的保修仍然有效，需要售后支持。
ai 
tool_calls: [{'name': 'record_warranty_status', 'args': {'status': 'valid'}, 'id': 'call_1', 'type': 'tool_call'}]
tool Warranty status recorded: valid
ai 已转到 specialist，并根据有效保修状态给出解决方案。
current_step: specialist
warranty_status: valid


## 6. 为什么需要 checkpointer

handoff 通常会改变 state。

如果后续对话还要保持在 specialist step，就需要保存 state。

这就是为什么示例里用了：

```python
checkpointer=InMemorySaver()
```

真实业务里应该使用持久化 checkpointer，并用 `thread_id` 绑定会话或流程实例。

## 7. Handoff 的设计要点

设计 handoff 时要明确：

1. 哪个状态字段表示 active agent / active step？
2. 哪个工具或条件触发 handoff？
3. handoff 后 prompt 是否变化？
4. handoff 后 tools 是否变化？
5. handoff 后是否需要保留前一阶段摘要？
6. 是否允许 handoff back？
7. 前端是否要展示当前 agent 名称？

如果这些问题没有设计清楚，handoff 会让用户感觉系统“突然换人但没人交接”。

## 8. 和本仓库 Harness 的对应关系

| LangChain Handoff | 本仓库 Harness 对应点 |
| --- | --- |
| `current_step` / active agent | planner action / role 状态 |
| `Command(update=...)` | 更新 ledger / completed_steps / active role |
| step-specific prompt | 不同 subagent prompt |
| step-specific tools | allowed_tools / allowed_paths |
| checkpointer + thread_id | approval ticket / session / run state |

区别是：

```text
Harness 当前更偏 coordinator 委派 subagent；
handoff 更像 active role 直接切换并接管后续对话。
```

## 9. 本讲练习

请判断下面场景更适合 subagent 还是 handoff：

1. 主 agent 需要调用 research agent 查资料，查完后主 agent 总结。
2. 用户从售前咨询进入合同条款讨论，后续都由合同 agent 处理。
3. 编码 agent 委派 verification agent 独立检查测试结果。
4. 客服机器人识别为技术故障后，后续由技术支持 agent 接管。

参考答案：

1. subagent
2. handoff
3. subagent
4. handoff

## 10. 本讲小结

这一讲的核心：

```text
Handoff 是 active agent / active step 的控制权转移。
```

你现在应该能判断：

- handoff 和 subagent 的区别
- 为什么 handoff 需要 state
- 为什么 handoff 后 prompt/tools 应该跟着变
- `Command(update=...)` 如何触发状态切换
- 为什么需要 checkpointer 保存切换后的状态

下一讲可以继续进入 Skills。